# Train YOLO-seg from image- and video-pipeline artifacts

Builds an **Ultralytics** segment dataset from runs produced by **`qwen_sam3_image_pipeline.ipynb`** and/or **`qwen_sam3_pipeline_video.ipynb`**. For each `labels/segments/<stem>.txt`, the **source RGB** is resolved in order: **(1)** `<run>/images/frames/<stem>.*` if it exists (video exports), else **(2)** `metadata.json` → **`image`** (image pipeline; often `.../all/inputs/...`).

- **Image pipeline — flat:** `ARTIFACTS_ROOT/<image_stem>/labels/segments/*.txt`
- **Image pipeline — nested (USS-PF):** `ARTIFACTS_ROOT/<seq_id>/<image_stem>/labels/segments/*.txt`
- **Video pipeline:** same run layout; matching frames are `.../images/frames/{06d}.jpg`. Use config **`EXPAND_VIDEO_FRAMES`**: `True` = one training row per frame/label, `False` = one label file per run.

Typical Drive layout: **labels** under `.../all/outputs/...`, **image-pipeline** photos from **`metadata['image']`**. If those paths still use a Colab prefix, set **`IMAGE_PATH_PREFIX_FROM` / `IMAGE_PATH_PREFIX_TO`** so the host prefix matches your machine; the `.../all/inputs/...` tail is unchanged.

**Outputs:** `DATASET_ROOT/` with `images/{train,val}/`, `labels/{train,val}/`, and `data.yaml`, then `yolo segment train` (or skip train for dataset-only runs).

**Device:** `DEVICE = "auto"` picks CUDA → MPS → CPU. **Dry run:** small subset + 1 epoch to verify the stack before a full run.

### Workflow: same Drive tree, local prep then Colab GPU

Keep **labels + `metadata.json`** on **Google Drive**; only the **absolute path prefix** changes between **mounted Drive on your Mac** and **Colab’s `/content/drive/...`**. Use one `ARTIFACTS_ROOT` / `DATASET_ROOT` pair per machine (see config cell comments). **`IMAGE_PATH_PREFIX_FROM` / `TO`** rewrite only the shared **`.../auto-labeler/all`** prefix inside `metadata["image"]` so `inputs/uss_pf/...` still resolves when metadata was written on the **other** host. If `image` paths already match the machine you are on, leave both prefixes **`None`**.

**Suggested:** materialize the dataset **locally** (or on Colab CPU into `/content/...`) for a **dry run / smoke train**; for the **long GPU run**, either train from Drive on Colab (slow I/O) or **zip `DATASET_ROOT`** after materialize, upload to Colab, unzip under **`/content/yolo_data`**, and point `data.yaml` `path:` there so training reads **local VM disk**.

#### Cell run order (0 = first code cell after this intro)

**On your Mac (prep dataset + optional smoke train)** — use **mounted Drive** paths in config; set `IMAGE_PATH_PREFIX_*` only if `metadata["image"]` still has Colab prefixes.

| Step | What to run | Notes |
|------|----------------|------|
| Install | Code cell `pip install ultralytics` | Once per venv / Colab session |
| (Optional) | Torch / CUDA probe cell | Skip on Mac if you like |
| Config | Big config cell | Mac `ARTIFACTS_ROOT` / `DATASET_ROOT`; **`RUN_TRAIN=False`** to only build data; **`USE_SYMLINK=False`** if you will **zip** the folder for Colab (symlinks do not pack real images) |
| Scan | `iter_segment_runs` + `collect_pairs` cell | |
| Unify | `CLASS_NAMES` / majority cell | |
| Split | `assign_splits` cell | |
| Build | **Materialize** cell | Writes `images/{train,val}/`, `labels/{train,val}/` |
| YAML | **Write `data.yaml`** cell | |
| Check | **Label sanity** cell | Must pass before any train |
| Local train | **Ultralytics `model.train`** cell | Optional smoke: set `RUN_TRAIN=True`, small `EPOCHS` / `DRY_RUN=True` |

**On Google Colab (GPU train)** — pick **one** path:

1. **Train from Drive (simplest, slower I/O):** Mount Drive, edit **config** to Colab `ARTIFACTS_ROOT` / `DATASET_ROOT` (under `/content/drive/...`). Set `IMAGE_PATH_PREFIX_*` **Mac → Colab** if metadata still has Mac paths. Re-run from **collect_pairs** through **label sanity**, then **`model.train`** (and optional val cell). Use **`USE_SYMLINK=True`** if images stay on Drive and paths resolve.

2. **Zip dataset (faster GPU I/O):** On Mac, prep through **label sanity** with **`USE_SYMLINK=False`**, zip **`DATASET_ROOT`**, upload to Colab, unzip e.g. `/content/yolo_seg_from_pipeline`. Edit **`data.yaml`** first line so `path:` is that `/content/...` folder (or re-run only the **write `data.yaml`** cell after changing `DATASET_ROOT` in config). Then run **pip** + **config** + **`model.train`** cell — you do **not** need collect/materialize again if the zip already contains `images/`, `labels/`, and `data.yaml`.


In [ ]:
# Optional: run once per environment
!pip install -q -U ultralytics pyyaml

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"PyTorch built with CUDA version: {torch.version.cuda}")
    print(f"Current CUDA device name: {torch.cuda.get_device_name(0)}")
    import subprocess
    try:
        nvcc_version = subprocess.check_output(['nvcc', '--version']).decode('utf-8')
        print(f"System NVCC CUDA version:\n{nvcc_version}")
    except FileNotFoundError:
        print("nvcc not found. Cannot determine system CUDA version via nvcc.")
    except Exception as e:
        print(f"Error getting nvcc version: {e}")

In [24]:
from __future__ import annotations

import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

# ── User config ─────────────────────────────────────────────────────────────
# Same Drive folder layout everywhere; only the *host prefix* changes (Mac vs Colab).
#
# Local Mac (Drive mounted — pick the path your Finder shows):
#   ARTIFACTS_ROOT = Path(".../Google Drive/My Drive/auto-labeler/all/outputs/uss_pf")
#   DATASET_ROOT   = Path(".../Google Drive/My Drive/auto-labeler/datasets/yolo_seg_from_pipeline")
# Google Colab (after drive.mount):
#   ARTIFACTS_ROOT = Path("/content/drive/MyDrive/auto-labeler/all/outputs/uss_pf")
#   # If your Drive shows "My Drive" with a space, try: /content/drive/My Drive/...
#   DATASET_ROOT   = Path("/content/drive/MyDrive/auto-labeler/datasets/yolo_seg_from_pipeline")
#
# Pipeline artifacts (labels, metadata) — USS-PF nested:
#   <ARTIFACTS_ROOT>/<seq_id>/<image_stem>/labels/segments/*.txt
ARTIFACTS_ROOT = Path("/Users/ral3ply/Google Drive/My Drive/auto-labeler/all/outputs/uss_pf")
# Or flat layout: ARTIFACTS_ROOT = Path(".../outputs/images")

# Built Ultralytics dataset (images + labels + data.yaml)
DATASET_ROOT = Path("/Users/ral3ply/Google Drive/My Drive/auto-labeler/datasets/yolo_seg_from_pipeline")

# How to place images into the dataset tree
USE_SYMLINK = False  # True = symlink (saves space); False = copy file

# Train/val
TRAIN_FRACTION = 0.85
RANDOM_SEED = 42
# "random" = per-image split | "sequence" = keep all frames from one leading seq_id in same split (nested only)
SPLIT_MODE = "sequence"

# Smoke test: limit number of samples (None = all). Ignored when DRY_RUN=True (uses DRY_RUN_MAX_SAMPLES).
MAX_SAMPLES: int | None = None  # e.g. 200 to cap; must be int or None (annotation-only line breaks comparisons)

# Skip runs with empty segment label file or missing source image
SKIP_EMPTY_LABELS = True
# qwen_sam3_pipeline_video: frames live under <run>/images/frames/<stem>.jpg. If True, one dataset row
# per segment .txt (all frames). If False, a single label per run (image-style: pick_segment_label_path).
EXPAND_VIDEO_FRAMES = True

# yolo_class_names must match across runs for one YOLO dataset. Open-vocab snapshots differ slightly between runs.
# "majority" = keep only runs whose class-name tuple matches the most common one (drops outliers).
# "strict" = original behavior: raise if any tuple differs (use after re-running standardize on ALL artifacts).
CLASS_NAMES_UNIFY_MODE = "majority"  # "strict" | "majority"

# metadata.json "image" → .../all/inputs/uss_pf/...jpg. Labels under .../all/outputs/uss_pf (ARTIFACTS_ROOT).
# Rewrite only the shared prefix up through .../auto-labeler/all (NOT .../outputs).
#
# On Mac, if metadata still has Colab paths:
#   FROM = "/content/drive/MyDrive/auto-labeler/all"   # or "/content/drive/My Drive/..."
#   TO   = "/Users/you/Google Drive/My Drive/auto-labeler/all"  # your mounted Drive
# On Colab, if metadata was written on Mac, swap FROM/TO.
# If metadata paths already exist on this machine, leave both None.
# When ARTIFACTS_ROOT is .../all/outputs/uss_pf, the scan cell also auto-maps metadata
# .../all/<tail> to this machine's .../all/<tail> (Colab ↔ Mac) without setting these.
IMAGE_PATH_PREFIX_FROM: str | None = None
IMAGE_PATH_PREFIX_TO: str | None = None
# macOS File Provider (path often under Library/CloudStorage/...):
# IMAGE_PATH_PREFIX_TO = str(next(Path("~/Library/CloudStorage").expanduser().glob("GoogleDrive-*")) / "My Drive/auto-labeler/all")

# ── Device (Ultralytics / PyTorch) ─────────────────────────────────────────
# "auto" → CUDA if available, else Apple MPS if available, else CPU.
# Force with "cuda", "mps", or "cpu".
DEVICE = "auto"

# ── Dry run (quick smoke: tiny data + 1 epoch) ─────────────────────────────
DRY_RUN = False  # True = small subset + 1 epoch + separate *_dryrun dataset folder
DRY_RUN_MAX_SAMPLES = 16
DRY_RUN_EPOCHS = 1
DRY_RUN_BATCH = 2
DRY_RUN_IMGSZ = 320

# ── Training (Ultralytics) ──────────────────────────────────────────────────
RUN_TRAIN = True  # False = only materialize dataset + data.yaml
YOLO_MODEL = "yolo26x-seg.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
PROJECT = str(DATASET_ROOT.parent / "yolo_seg_runs")
NAME = "from_image_pipeline"

# Isolate dry-run files so you do not wipe a full dataset tree
if DRY_RUN:
    DATASET_ROOT = DATASET_ROOT.parent / f"{DATASET_ROOT.name}_dryrun"
    NAME = f"{NAME}_dryrun"

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT.resolve())
print("DATASET_ROOT:  ", DATASET_ROOT.resolve())
print(
    f"DEVICE={DEVICE!r}  DRY_RUN={DRY_RUN}  RUN_TRAIN={RUN_TRAIN}  "
    f"EXPAND_VIDEO_FRAMES={EXPAND_VIDEO_FRAMES}"
)
if IMAGE_PATH_PREFIX_FROM and IMAGE_PATH_PREFIX_TO:
    print("IMAGE path rewrite:", repr(IMAGE_PATH_PREFIX_FROM), "->", Path(IMAGE_PATH_PREFIX_TO).expanduser())


ARTIFACTS_ROOT: /Users/ral3ply/Library/CloudStorage/GoogleDrive-ali.rahim.taleqani@gmail.com/My Drive/auto-labeler/all/outputs/uss_pf
DATASET_ROOT:   /Users/ral3ply/Library/CloudStorage/GoogleDrive-ali.rahim.taleqani@gmail.com/My Drive/auto-labeler/datasets/yolo_seg_from_pipeline
DEVICE='auto'  DRY_RUN=False  RUN_TRAIN=True


In [25]:
def iter_segment_runs(artifacts_root: Path):
    """Yield (sample_key, run_dir) for each folder that has labels/segments."""
    if not artifacts_root.is_dir():
        raise FileNotFoundError(artifacts_root)
    for sub in sorted(artifacts_root.iterdir()):
        if not sub.is_dir() or sub.name.startswith("."):
            continue
        if (sub / "labels" / "segments").is_dir():
            yield sub.name, sub
            continue
        for nest in sorted(sub.iterdir()):
            if not nest.is_dir() or nest.name.startswith("."):
                continue
            if (nest / "labels" / "segments").is_dir():
                yield f"{sub.name}__{nest.name}", nest


def pick_segment_label_path(run_dir: Path) -> Path | None:
    seg = run_dir / "labels" / "segments"
    cand = [p for p in seg.glob("*.txt") if p.name != "classes.txt"]
    if not cand:
        return None
    preferred = seg / f"{run_dir.name}.txt"
    if preferred.is_file():
        return preferred
    return sorted(cand)[0]


def read_classes_from_run(run_dir: Path) -> list[str] | None:
    p = run_dir / "labels" / "segments" / "classes.txt"
    if not p.is_file():
        return None
    lines = [x.strip() for x in p.read_text().splitlines() if x.strip()]
    return lines if lines else None


def resolve_metadata_image_path(raw: str) -> Path:
    """Use metadata path as-is, or rewrite prefix if images live under a different root."""
    p = Path(raw)
    if p.is_file():
        return p.resolve()
    fa, tb = IMAGE_PATH_PREFIX_FROM, IMAGE_PATH_PREFIX_TO
    if fa and tb and raw.startswith(fa):
        suffix = raw[len(fa) :].lstrip("/\\")
        cand = Path(tb).expanduser() / suffix
        if cand.is_file():
            return cand.resolve()
    # Auto-rewrite when metadata was written on another host (e.g. Colab vs Mac): same
    # .../<project>/all/<tail> layout; local ALL root = parent of `outputs/` (ARTIFACTS_ROOT..outputs..uss_pf).
    try:
        norm = raw.replace("\\", "/")
        marker = "/all/"
        if marker not in norm:
            raise ValueError("no /all/ segment")
        i = norm.index(marker)
        remote_all = norm[:i] + "/all"
        tail = norm[len(remote_all) :].lstrip("/")
        local_all = ARTIFACTS_ROOT.parent.parent
        if local_all.is_dir() and tail:
            cand = local_all / tail
            if cand.is_file():
                return cand.resolve()
    except (ValueError, OSError):
        pass
    return p


_FRAME_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def list_segment_labels(run_dir: Path) -> list[Path]:
    seg = run_dir / "labels" / "segments"
    if not seg.is_dir():
        return []
    return sorted(p for p in seg.glob("*.txt") if p.name != "classes.txt")


def resolve_source_image_path(run_dir: Path, meta: dict, lbl: Path) -> Path | None:
    """
    Source RGB for a segment label, same idea as image + video pipelines.
    1) qwen_sam3_pipeline_video: <run_dir>/images/frames/<label_stem>.<ext>
    2) qwen_sam3_image_pipeline: metadata['image'] (prefix rewrite via resolve_metadata_image_path)
    """
    for ext in _FRAME_EXTS:
        c = run_dir / "images" / "frames" / f"{lbl.stem}{ext}"
        if c.is_file():
            return c.resolve()
    raw = meta.get("image")
    if not raw:
        return None
    p = resolve_metadata_image_path(str(raw))
    return p if p.is_file() else None


def collect_pairs(artifacts_root: Path):
    rows = []
    expand = bool(globals().get("EXPAND_VIDEO_FRAMES", True))
    for key, run_dir in iter_segment_runs(artifacts_root):
        meta_path = run_dir / "labels" / "metadata.json"
        if not meta_path.is_file():
            rows.append(
                {"key": key, "run_dir": str(run_dir), "ok": False, "reason": "no metadata.json"}
            )
            continue
        meta = json.loads(meta_path.read_text())
        names = meta.get("yolo_class_names") or read_classes_from_run(run_dir)
        if not names:
            rows.append(
                {"key": key, "run_dir": str(run_dir), "ok": False, "reason": "no class names"}
            )
            continue

        all_lbls = list_segment_labels(run_dir)
        if not all_lbls:
            rows.append(
                {"key": key, "run_dir": str(run_dir), "ok": False, "reason": "no segment txt"}
            )
            continue

        frames_dir = run_dir / "images" / "frames"
        if expand and frames_dir.is_dir():
            lbls_to_use = all_lbls
        else:
            pl = pick_segment_label_path(run_dir)
            lbls_to_use = [pl] if pl is not None else []

        if not lbls_to_use:
            rows.append(
                {"key": key, "run_dir": str(run_dir), "ok": False, "reason": "no segment label"}
            )
            continue
        multi = len(lbls_to_use) > 1

        for lbl in lbls_to_use:
            row_key = f"{key}__f_{lbl.stem}" if multi else key
            raw = lbl.read_text().strip() if lbl.is_file() else ""
            if SKIP_EMPTY_LABELS and not raw:
                rows.append(
                    {
                        "key": row_key,
                        "run_dir": str(run_dir),
                        "ok": False,
                        "reason": "empty label",
                    }
                )
                continue
            img = resolve_source_image_path(run_dir, meta, lbl)
            if img is None:
                rows.append(
                    {
                        "key": row_key,
                        "run_dir": str(run_dir),
                        "ok": False,
                        "reason": (
                            f"no source image for {lbl.name} "
                            f"(tried {run_dir / 'images' / 'frames' / (lbl.stem + '.*')}, "
                            "then metadata['image'])"
                        ),
                    }
                )
                continue
            rows.append(
                {
                    "key": row_key,
                    "run_dir": str(run_dir),
                    "label_path": str(lbl),
                    "image_path": str(img),
                    "nc": len(names),
                    "class_names": names,
                    "ok": True,
                }
            )
    return rows


rows = collect_pairs(ARTIFACTS_ROOT)
ok = [r for r in rows if r["ok"]]
bad = [r for r in rows if not r["ok"]]
print(f"Runs scanned: {len(rows)}  usable: {len(ok)}  skipped: {len(bad)}  EXPAND_VIDEO_FRAMES={EXPAND_VIDEO_FRAMES}")
if bad[:8]:
    print("Example skips:")
    for r in bad[:8]:
        print(f"  {r.get('key')}: {r.get('reason')}")
if not ok:
    _la = ARTIFACTS_ROOT.parent.parent
    raise RuntimeError(
        "No usable samples — check ARTIFACTS_ROOT, labels/metadata.json, and source images. "
        "Video runs: <run>/images/frames/<stem>.* (qwen_sam3_pipeline_video). "
        "Image runs: metadata['image'] paths under .../all/inputs/... (qwen_sam3_image_pipeline). "
        f"Inferred ALL root: {_la}. Set IMAGE_PATH_PREFIX_FROM/TO if Colab vs Mac prefixes differ."
    )


Runs scanned: 1980  usable: 1972  skipped: 8
Example skips:
  107111__20190306_LB_XO2617_002_172506_Front.avi: empty label
  107449__20190612_LB_XO2617_165908_front.avi: no segment txt
  108367__20190613_LB_XO7246_090231_front.avi: empty label
  108367__20190613_LB_XO7246_090231_rear.avi: empty label
  108373__20190613_LB_XO7246_090013_front.avi: empty label
  108373__20190613_LB_XO7246_090013_rear.avi: empty label
  108389__20190613_LB_XO7246_085909_front.avi: empty label
  108389__20190613_LB_XO7246_085909_rear.avi: empty label


In [26]:
# Subsample + class-name consistency (DRY_RUN forces a small cap)
from collections import Counter


def unify_class_names(ok_rows: list, mode: str) -> list:
    if mode != "majority" or not ok_rows:
        return ok_rows
    counts = Counter(tuple(r["class_names"]) for r in ok_rows)
    canon, nkeep = counts.most_common(1)[0]
    out = [r for r in ok_rows if tuple(r["class_names"]) == canon]
    dropped = len(ok_rows) - len(out)
    if dropped:
        print(
            f"CLASS_NAMES_UNIFY_MODE='majority': using {len(canon)} classes, {nkeep} runs match; "
            f"dropped {dropped} run(s) with other vocab snapshots. For zero drops, re-run standardize_open_vocab_extend_yolo on all of ARTIFACTS_ROOT."
        )
    return out


ok = unify_class_names(ok, CLASS_NAMES_UNIFY_MODE)
if not ok:
    raise RuntimeError("No runs left after class-name unify — set CLASS_NAMES_UNIFY_MODE='strict' and fix metadata, or check ARTIFACTS_ROOT.")

if DRY_RUN:
    cap = min(DRY_RUN_MAX_SAMPLES, len(ok))
    if len(ok) > cap:
        rng = random.Random(RANDOM_SEED)
        ok = rng.sample(ok, cap)
    print(f"DRY_RUN: using {len(ok)} samples (cap={DRY_RUN_MAX_SAMPLES})")
elif MAX_SAMPLES is not None and len(ok) > MAX_SAMPLES:
    rng = random.Random(RANDOM_SEED)
    ok = rng.sample(ok, MAX_SAMPLES)
    print(f"Subsampled to MAX_SAMPLES={MAX_SAMPLES}")

if DRY_RUN:
    EPOCHS = int(DRY_RUN_EPOCHS)
    BATCH = int(DRY_RUN_BATCH)
    IMGSZ = int(DRY_RUN_IMGSZ)
    print(f"DRY_RUN training overrides: epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ}")

# All usable rows must share same nc and identical class name list (order matters)
ncs = {tuple(r["class_names"]) for r in ok}
if len(ncs) != 1:
    raise ValueError(
        f"Inconsistent yolo_class_names across runs: {len(ncs)} variants (show first 3): {list(ncs)[:3]}. "
        "Set CLASS_NAMES_UNIFY_MODE = 'majority' in the config cell, or re-run standardize on every run so metadata matches."
    )

CLASS_NAMES = list(ok[0]["class_names"])
NC = len(CLASS_NAMES)
print(f"NC={NC} names={CLASS_NAMES}")


NC=284 names=['person', 'car', 'van', 'bus', 'truck', 'cone', 'sign', 'pole', 'barrier', 'vegetation', 'road', 'sidewalk', 'building', 'wall', 'overhead', 'parking lot', 'sky', 'planter', 'stairs', 'floor', 'fence', 'structure', 'ceiling', 'window blinds', 'trash bin', 'railing', 'road marking', 'bollard', 'carport', 'concrete pillars', 'box', 'cart', 'concrete overpass', 'concrete planter', 'gravel', 'scene artifact', 'carpet', 'airport terminal', 'concrete surface', 'vehicle part', 'ego interior', 'air conditioning unit', 'arrow', 'surface', 'fountain', 'train tracks', 'tree trunk', 'advertisement', 'concrete paving stones', 'train window', 'sunlight reflection', 'fire hydrant', 'skateboard deck', 'roof structure', 'ground surface', 'pile of debris', 'garage doors', 'concrete driveway', 'stone wall', 'barrel', 'asphalt ground', 'manhole cover', 'metal gate', 'drain', 'pen', 'rock wall', 'crack', 'wooden pallets', 'concrete ground', 'window panes', 'car windshield wiper', 'wooden blin

In [27]:
def sequence_group_key(row: dict) -> str:
    """For nested keys like '106041__stem', use leading seq; else whole key."""
    k = row["key"]
    if "__" in k:
        return k.split("__", 1)[0]
    return k


def assign_splits(ok_rows: list, mode: str, train_frac: float, seed: int) -> dict[str, str]:
    """Return map row key -> 'train'|'val'."""
    rng = random.Random(seed)
    out = {}
    if mode == "sequence":
        groups = defaultdict(list)
        for r in ok_rows:
            groups[sequence_group_key(r)].append(r)
        gkeys = list(groups.keys())
        rng.shuffle(gkeys)
        n_train = int(round(len(gkeys) * train_frac))
        train_groups = set(gkeys[:n_train])
        for gk, rs in groups.items():
            split = "train" if gk in train_groups else "val"
            for r in rs:
                out[r["key"]] = split
        return out
    # random per image
    keys = [r["key"] for r in ok_rows]
    rng.shuffle(keys)
    n_train = int(round(len(keys) * train_frac))
    train_k = set(keys[:n_train])
    for r in ok_rows:
        out[r["key"]] = "train" if r["key"] in train_k else "val"
    return out


split_of = assign_splits(ok, SPLIT_MODE, TRAIN_FRACTION, RANDOM_SEED)
n_tr = sum(1 for s in split_of.values() if s == "train")
n_va = sum(1 for s in split_of.values() if s == "val")
n_tot = n_tr + n_va
print(f"SPLIT_MODE={SPLIT_MODE!r}  images: train={n_tr}  val={n_va}  (total {n_tot})")
if n_tot:
    print(f"  fractions: train={n_tr / n_tot:.1%}  val={n_va / n_tot:.1%}  (target TRAIN_FRACTION={TRAIN_FRACTION})")
if SPLIT_MODE == "sequence":
    g_tr = {sequence_group_key(r) for r in ok if split_of[r["key"]] == "train"}
    g_va = {sequence_group_key(r) for r in ok if split_of[r["key"]] == "val"}
    print(f"  sequence groups (USS id): train={len(g_tr)}  val={len(g_va)} — all frames in one group share the same split")


SPLIT_MODE='sequence'  images: train=1677  val=295  (total 1972)
  fractions: train=85.0%  val=15.0%  (target TRAIN_FRACTION=0.85)
  sequence groups (USS id): train=839  val=148 — all frames in one group share the same split


In [28]:
# Materialize dataset: clear previous train/val trees (keep other files if any)
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    p = DATASET_ROOT / sub
    if p.is_dir():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)


def link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if USE_SYMLINK:
        dst.symlink_to(src.resolve())
    else:
        shutil.copy2(src, dst)


# Use stable filesystem-safe ids
def sample_id(row: dict) -> str:
    return row["key"].replace("/", "_").replace(" ", "_")


written = []
for r in ok:
    sid = sample_id(r)
    sp = split_of[r["key"]]
    ext = Path(r["image_path"]).suffix.lower() or ".jpg"
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
        ext = ".jpg"
    img_dst = DATASET_ROOT / "images" / sp / f"{sid}{ext}"
    lbl_dst = DATASET_ROOT / "labels" / sp / f"{sid}.txt"
    link_or_copy(Path(r["image_path"]), img_dst)
    link_or_copy(Path(r["label_path"]), lbl_dst)
    written.append({"id": sid, "split": sp})

print(f"Materialized {len(written)} image/label pairs under {DATASET_ROOT}")


Materialized 1972 image/label pairs under /Users/ral3ply/Google Drive/My Drive/auto-labeler/datasets/yolo_seg_from_pipeline


In [29]:
# Write data.yaml (Ultralytics segment)
names_dict = {i: n for i, n in enumerate(CLASS_NAMES)}
yaml_lines = [
    f"# Auto-generated from image pipeline artifacts\n",
    # Ultralytics resolves `path` relative to this yaml's directory (see ultralytics data utils).
    # Using `.` keeps the same file valid on Mac, Colab (/content/drive/...), or any mount.
    "path: .\n",
    "train: images/train\n",
    "val: images/val\n",
    f"nc: {NC}\n",
    "names:\n",
]
for i, n in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {i}: {n}\n")

data_yaml = DATASET_ROOT / "data.yaml"
data_yaml.write_text("".join(yaml_lines))
print("Wrote", data_yaml)
print(data_yaml.read_text())


Wrote /Users/ral3ply/Google Drive/My Drive/auto-labeler/datasets/yolo_seg_from_pipeline/data.yaml
# Auto-generated from image pipeline artifacts
path: .
train: images/train
val: images/val
nc: 284
names:
  0: person
  1: car
  2: van
  3: bus
  4: truck
  5: cone
  6: sign
  7: pole
  8: barrier
  9: vegetation
  10: road
  11: sidewalk
  12: building
  13: wall
  14: overhead
  15: parking lot
  16: sky
  17: planter
  18: stairs
  19: floor
  20: fence
  21: structure
  22: ceiling
  23: window blinds
  24: trash bin
  25: railing
  26: road marking
  27: bollard
  28: carport
  29: concrete pillars
  30: box
  31: cart
  32: concrete overpass
  33: concrete planter
  34: gravel
  35: scene artifact
  36: carpet
  37: airport terminal
  38: concrete surface
  39: vehicle part
  40: ego interior
  41: air conditioning unit
  42: arrow
  43: surface
  44: fountain
  45: train tracks
  46: tree trunk
  47: advertisement
  48: concrete paving stones
  49: train window
  50: sunlight refl

In [30]:
# Label sanity before train (avoids cryptic IndexError in val/plots: class id >= nc)
issues = 0
max_cid = -1
for r in ok:
    sid = sample_id(r)
    sp = split_of[r["key"]]
    txt = (DATASET_ROOT / "labels" / sp / f"{sid}.txt").read_text().strip()
    if not txt:
        issues += 1
        continue
    bad_row = False
    for li, line in enumerate(txt.splitlines(), 1):
        parts = line.split()
        if len(parts) < 7:
            issues += 1
            bad_row = True
            break
        try:
            cid = int(float(parts[0]))
        except ValueError:
            issues += 1
            bad_row = True
            break
        max_cid = max(max_cid, cid)
        if cid < 0 or cid >= NC:
            issues += 1
            bad_row = True
            break
print(f"Label sanity: {issues} bad file(s), max class id in materialized labels={max_cid}, NC={NC}")
if max_cid >= NC:
    raise ValueError(
        f"Class id {max_cid} appears in segment labels but NC={NC} (len(CLASS_NAMES)). "
        "Ultralytics then crashes (e.g. index 15 vs axis size 7). Fix: use the same extended "
        "vocab on every run under ARTIFACTS_ROOT, re-run standardize_open_vocab_extend_yolo / remap, "
        "and re-materialize this dataset so yolo_class_names and every .txt match."
    )
if issues:
    raise RuntimeError(
        f"Label sanity failed ({issues} file(s)). Fix segment lines (class + polygon) before training."
    )


Label sanity: 0 bad file(s), max class id in materialized labels=282, NC=284


In [31]:
# Materialized pair sanity: every .txt has a real image on disk (catches broken Drive symlinks / ENOENT on Colab)
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
VERIFY_IMAGE_DECODE = True  # True = PIL verify each file (slower, catches corrupt bytes)


def _find_image_for_label(images_dir: Path, stem: str) -> Path | None:
    if not images_dir.is_dir():
        return None
    for ext in IMAGE_EXTS:
        p = images_dir / f"{stem}{ext}"
        if p.is_file():
            return p
    for p in sorted(images_dir.glob(f"{stem}.*")):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            return p
    return None


def _image_path_report(img: Path | None) -> str | None:
    if img is None:
        return "no_matching_image"
    try:
        if not img.exists():
            return "missing_or_broken_symlink"
        if not img.is_file():
            return "not_a_regular_file"
        if img.stat().st_size == 0:
            return "zero_bytes"
    except OSError as e:
        return f"os_error:{e}"
    if VERIFY_IMAGE_DECODE:
        from PIL import Image

        try:
            with Image.open(img) as im:
                im.verify()
        except Exception as e:
            return f"decode_error:{type(e).__name__}:{e}"
    return None


pair_issues: list[tuple[str, str, str]] = []
label_parse_issues: list[tuple[str, str, str]] = []
max_cid_scan = -1

for split in ("train", "val"):
    ld = DATASET_ROOT / "labels" / split
    idir = DATASET_ROOT / "images" / split
    if not ld.is_dir():
        pair_issues.append((split, "__dir__", f"missing labels dir {ld}"))
        continue
    for lp in sorted(ld.glob("*.txt")):
        stem = lp.stem
        img = _find_image_for_label(idir, stem)
        reason = _image_path_report(img)
        if reason:
            pair_issues.append((split, stem, reason))
        try:
            txt = lp.read_text().strip()
        except OSError as e:
            label_parse_issues.append((split, stem, f"read_error:{e}"))
            continue
        if not txt:
            label_parse_issues.append((split, stem, "empty_label"))
            continue
        for line in txt.splitlines():
            parts = line.split()
            if len(parts) < 7:
                label_parse_issues.append((split, stem, "bad_segment_line"))
                break
            try:
                cid = int(float(parts[0]))
            except ValueError:
                label_parse_issues.append((split, stem, "bad_class_token"))
                break
            max_cid_scan = max(max_cid_scan, cid)
            if cid < 0 or cid >= NC:
                label_parse_issues.append((split, stem, f"class_out_of_range:{cid}"))
                break

orphan_images: list[tuple[str, str]] = []
for split in ("train", "val"):
    idir = DATASET_ROOT / "images" / split
    ld = DATASET_ROOT / "labels" / split
    if not idir.is_dir():
        continue
    for ip in sorted(idir.iterdir()):
        if not ip.is_file() or ip.suffix.lower() not in IMAGE_EXTS:
            continue
        if not (ld / f"{ip.stem}.txt").is_file():
            orphan_images.append((split, ip.name))

print(
    f"Pair sanity: {len(pair_issues)} image/path issues, {len(label_parse_issues)} label issues, "
    f"{len(orphan_images)} images without matching .txt, max_cid={max_cid_scan} vs NC={NC}"
)
for row in pair_issues[:25]:
    print(" ", row)
if len(pair_issues) > 25:
    print(f"  ... and {len(pair_issues) - 25} more image/path issues")
for row in label_parse_issues[:25]:
    print(" ", row)
if len(label_parse_issues) > 25:
    print(f"  ... and {len(label_parse_issues) - 25} more label issues")
if orphan_images:
    print(f"Note: {len(orphan_images)} image(s) have no label (YOLO may ignore them): examples", orphan_images[:5])

if max_cid_scan >= NC:
    raise ValueError(
        f"max class id in on-disk labels is {max_cid_scan} but NC={NC}. Re-sync class list and re-materialize."
    )
if pair_issues or label_parse_issues:
    raise RuntimeError(
        "Pair sanity failed: fix missing/broken image paths (often USE_SYMLINK on Colab) or bad labels, then re-run materialize."
    )


Pair sanity: 0 image/path issues, 0 label issues, 0 images without matching .txt, max_cid=282 vs NC=284


In [ ]:
def resolve_training_device(preference: str):
    """Return a value Ultralytics accepts: 0 / 'mps' / 'cpu'."""
    import torch

    p = (preference or "auto").strip().lower()
    if p == "auto":
        if torch.cuda.is_available():
            return 0
        if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
            return "mps"
        return "cpu"
    if p == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError('DEVICE is "cuda" but torch.cuda.is_available() is False')
        return 0
    if p == "mps":
        if not (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()):
            raise RuntimeError('DEVICE is "mps" but MPS is not available (need Apple Silicon + recent PyTorch)')
        return "mps"
    if p == "cpu":
        return "cpu"
    raise ValueError(f"Unknown DEVICE={preference!r} (use auto|cuda|mps|cpu)")


if RUN_TRAIN:
    from ultralytics import YOLO

    train_device = resolve_training_device(DEVICE)
    print(f"Training device resolved to: {train_device!r}")

    model = YOLO(YOLO_MODEL)
    results = model.train(
        data=str(data_yaml.resolve()),
        epochs=int(EPOCHS),
        imgsz=int(IMGSZ),
        batch=int(BATCH),
        device=train_device,
        project=PROJECT,
        name=NAME,
    )
    print("Training finished; best weights under project directory.")
else:
    print(
        "RUN_TRAIN=False — dataset ready. Example:\n"
        f"  yolo segment train data={data_yaml} model=yolo11n-seg.pt epochs=1 imgsz=640 device=mps"
    )


In [ ]:
# --- After training: val + predictions (visual sanity) ---
# Uses best.pt from this NAME, or auto-picks newest run under PROJECT whose nc matches data.yaml.
# Optional: set WEIGHTS_PATH to a specific best.pt if you have multiple runs.
from pathlib import Path

import yaml
from ultralytics import YOLO

VAL_PLOTS = False  # True: confusion-matrix plots (needs model.nc == dataset classes)
WEIGHTS_PATH: Path | None = None  # e.g. Path(PROJECT) / "from_image_pipeline_dryrun2" / "weights" / "best.pt"


def _load_nc(path: Path) -> tuple[YOLO, int]:
    model = YOLO(str(path))
    nc = int(getattr(model.model, "nc", None) or len(model.names or {}))
    return model, nc


def resolve_best_weights(project_dir: Path, preferred: Path, nc_target: int) -> tuple[Path, YOLO, int]:
    """Prefer `preferred`; else newest */weights/best.pt under project with model.nc == nc_target."""
    if preferred.is_file():
        m, nc = _load_nc(preferred)
        if nc == nc_target:
            return preferred, m, nc
        print(f"Preferred weights nc={nc} != data nc={nc_target}, scanning {project_dir} ...")
    for p in sorted(project_dir.glob("*/weights/best.pt"), key=lambda x: x.stat().st_mtime, reverse=True):
        m, nc = _load_nc(p)
        if nc == nc_target:
            print("Using weights (nc match, newest first):", p.resolve())
            return p, m, nc
    if preferred.is_file():
        m, nc = _load_nc(preferred)
        return preferred, m, nc
    raise FileNotFoundError(f"No weights at {preferred} and no */weights/best.pt under {project_dir}")


preferred = WEIGHTS_PATH or (Path(PROJECT) / NAME / "weights" / "best.pt")
dev = resolve_training_device(DEVICE)
dy = yaml.safe_load(data_yaml.read_text())
nc_yaml = int(dy.get("nc") or len(dy.get("names") or []))
print("data.yaml NC:", nc_yaml, "|", data_yaml.resolve())

weights, m, nc_model = resolve_best_weights(Path(PROJECT), preferred, nc_yaml)
print("Weights:", weights.resolve(), "| model.nc:", nc_model)

val_verbose = nc_model == nc_yaml
if not val_verbose:
    print(
        "Warning: model.nc still != data.yaml nc — val per-class log would KeyError; running val with verbose=False. "
        "Fix: set WEIGHTS_PATH to the best.pt from the run that trained on this data.yaml, or align NAME with that run."
    )

metrics = m.val(
    data=str(data_yaml.resolve()),
    imgsz=int(IMGSZ),
    device=dev,
    plots=VAL_PLOTS,
    verbose=val_verbose,
)
rd = getattr(metrics, "results_dict", None) or getattr(metrics, "mean_results", None)
print("Val metrics (dict or tuple):", rd)

val_dir = DATASET_ROOT / "images" / "val"
pred_name = f"{weights.parent.parent.name}_predict_smoke"
# Default predict conf=0.25 hides weak heads; smoke runs often need lower conf to see *anything*.
PREDICT_CONF = 0.05
if nc_model != nc_yaml:
    print("Skipping predict: outputs would not match this data.yaml until weights nc matches.")
elif val_dir.is_dir() and any(val_dir.iterdir()):
    pred_results = m.predict(
        source=str(val_dir),
        imgsz=int(IMGSZ),
        device=dev,
        conf=float(PREDICT_CONF),
        save=True,
        project=str(Path(PROJECT)),
        name=pred_name,
    )
    out_dir = Path(PROJECT) / pred_name
    n_dets = sum(len(r.boxes) if r.boxes is not None else 0 for r in pred_results)
    print("Saved images under:", out_dir)
    print(
        f"Total boxes drawn (conf≥{PREDICT_CONF}): {n_dets}. "
        "If 0, files still look like plain camera frames — the head is not firing yet (try more epochs / data, "
        "or temporarily PREDICT_CONF=0.01)."
    )
else:
    print("No val images in", val_dir)
